In [191]:
#Capstone Project - Cease and Desist Doc Processing
from google.colab import drive
drive.mount('/content/drive' , force_remount = True)

Mounted at /content/drive


In [213]:
import os
import sys
folder_path = "/content/drive/MyDrive/pdf"


In [11]:
#these are already installed
!pip uninstall -y langchain langchain-core langchain-community langgraph langchain-groq pydantic requests tenacity


In [12]:
!pip install \
  "pydantic==2.12.3" \
  "requests==2.32.4" \
  "tenacity==8.5.0" \
  "langchain==0.2.16" \
  "langchain-core==0.2.43" \
  "langchain-community==0.2.16" \
  "langgraph==0.0.55" \
  "langchain-groq==0.1.9"

  Using cached pydantic-2.12.3-py3-none-any.whl.metadata (87 kB)
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached tenacity-8.5.0-py3-none-any.whl.metadata (1.2 kB)
  Using cached langchain-0.2.16-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_core-0.2.43-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_community-0.2.16-py3-none-any.whl.metadata (2.7 kB)
  Using cached langgraph-0.0.55-py3-none-any.whl.metadata (23 kB)
  Using cached langchain_groq-0.1.9-py3-none-any.whl.metadata (2.9 kB)
  Using cached pydantic_core-2.41.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached langchain_text_splitters-0.2.4-py3-none-any.whl.metadata (2.3 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.4/462.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 7.4 MB/s eta 0:00:00
Using cached tenacity-8.5.0-py3-

In [214]:
!apt-get install -y tesseract-ocr
!apt-get install -y poppler-utils
!pip install pytesseract pdf2image
!pip install pypdf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 6 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 6 not upgraded.


In [215]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

In [216]:
import os, json, re, sqlite3
from typing import TypedDict, Dict, Any

#from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
#from langgraph.types import interrupt

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_community.document_loaders import PyPDFLoader


In [217]:
##llm model
from langchain_groq import ChatGroq
MODEL_NAME = "llama-3.1-8b-instant"

llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name=MODEL_NAME,
    temperature=0
)

In [218]:
##safe llm rate limit
LAST_CALL = 0
MIN_DELAY = 2.5

def throttle():
    global LAST_CALL
    now = time.time()
    gap = now - LAST_CALL

    if gap < MIN_DELAY:
        time.sleep(MIN_DELAY - gap)

    LAST_CALL = time.time()


def safe_llm(prompt, retries=3):
    for i in range(retries):
        try:
            throttle()
            return llm.invoke(prompt).content
        except Exception as e:
            if "429" in str(e):
                wait = 4 + i * 2
                print(f"Rate limit → retry in {wait}s")
                time.sleep(wait)
            else:
                raise e
    return "Error"

In [219]:
##PDF loader and OCR for scan image

def run_ocr(file_path):
    print(f"OCR running: {file_path}")

    images = convert_from_path(file_path, dpi=300)
    text = ""

    for img in images:
        txt = pytesseract.image_to_string(
            img,
            config="--oem 3 --psm 6"
        )
        text += txt + "\n"

    # STRUCTURE (IMPORTANT)
    text = re.sub(r'[^\w\s:/\-\.,$()&\n]', ' ', text)
    text = re.sub(r'\n+', '\n', text)   #  line breaks
    text = re.sub(r' +', ' ', text)

    return text

def load_all_pdfs(folder_path):
    docs = []

    for file in os.listdir(folder_path):
        if file.lower().endswith(".pdf"):
            path = os.path.join(folder_path, file)

            print(f"\n📄 Loading: {file}")

            loader = PyPDFLoader(path)
            pages = loader.load()
            text = "\n".join([p.page_content for p in pages])


            if len(text.strip()) < 100 or len(text.split()) < 30:
                print("Low text → using OCR")
                text = run_ocr(path)

            docs.append({"file": file, "text": text})

    return docs

In [220]:
##classfication of LOA, NOTICE and Business document

@tool
def classify_document(text: str) -> str:
    """Classify document into LOA, NOTICE, BUSINESS with strict confidence."""

    return safe_llm(f"""
Classify the document STRICTLY:

---------------- DEFINITIONS ----------------

LOA (Letter of Authorization / Power of Attorney):
- Contains phrases like:
  * "Power of Attorney"
  * "authorize"
  * "retained the services of"
- One party gives authority to another
- Mentions agent, attorney, or law firm

NOTICE:
- Communication, warning, or request
- Has sender and recipient

BUSINESS:
- General agreements, contracts, invoices
- NO authorization relationship

---------------- PRIORITY RULE ----------------
If document contains ANY of:
- "Power of Attorney"
- "authorize"
- "retained the services"


---------------- OUTPUT FORMAT ----------------

Type: LOA or NOTICE or BUSINESS
Confidence: <0 to 1>

---------------- DOCUMENT ----------------
{text[:1200]}
""")

In [221]:
def parse_classification(output):
    t = re.search(r"(LOA|NOTICE|BUSINESS)", output, re.I)
    c = re.search(r"confidence\s*[:\-]?\s*(0?\.\d+|1(?:\.0)?)", output, re.I)

    return (
        t.group(1).upper() if t else "UNKNOWN",
        float(c.group(1)) if c else 0.5
    )

In [222]:
def rule_based_classification(text):

    loa_keywords = [
        "power of attorney", "authorize", "authorized",
        "retained the services", "agent", "attorney"
    ]

    notice_keywords = [
        "notice", "dear", "to whom it may concern",
        "please be advised", "inform", "request"
    ]

    business_keywords = [
        "agreement", "contract", "invoice",
        "payment", "terms", "amount"
    ]

    loa_score = sum(1 for k in loa_keywords if k in text)
    notice_score = sum(1 for k in notice_keywords if k in text)
    business_score = sum(1 for k in business_keywords if k in text)

    scores = {
        "LOA": loa_score,
        "NOTICE": notice_score,
        "BUSINESS": business_score
    }

    best = max(scores, key=scores.get)
    score = scores[best]

    if score == 0:
        return "UNKNOWN", 0.5

    return best, min(0.95, 0.6 + score * 0.1)

In [223]:
@tool
def extract_loa(text: str) -> str:
    """Extract LOA (Letter of Authorization) structured data with high precision."""

    return safe_llm(f"""
You are a HIGH PRECISION legal extraction system.

Extract ONLY structured data for LOA.

Return STRICT JSON:

{{
  "Authorizing Party": "",
  "Authorized Party": "",
  "Authorization Scope": [],
  "Effective Date": "",
  "Signature": ""
}}

---------------- HARD RULES ----------------

Authorizing Party:
- MUST be REAL PERSON NAME
- NEVER return:
  "I", "we", "I (we)", "client"
- Extract from:
  * Signature
  * Printed name near signature
  * Client Name field
- If not found → ""

Authorized Party:
- MUST be COMPANY / LAW FIRM
- MUST contain: LLC / LAW / GROUP / FIRM / LLP

Signature:
- ONLY real names
- If duplicate of Authorizing Party → keep BOTH but clean

DO NOT:
- Guess
- Hallucinate
- Swap fields

---------------- DOCUMENT ----------------
{text[:2500]}
""")

In [224]:
@tool
def extract_notice(text: str) -> str:
    """Extract Notice document structured data."""
    return safe_llm(f"""
You are a HIGH PRECISION notice extraction system.

Return ONLY valid JSON:

{{
  "Notice Type": "",
  "Sender": "",
  "Recipient": "",
  "Subject": "",
  "Important Dates": [],
  "Action Required": []
}}

---------------- STRICT RULES ----------------

Sender:
- MUST be person or organization
- Extract from signature or header
- If not found → "Not specified"

Recipient:
- Extract from:
  * "To:"
  * "Dear"
- If not found → "Not specified"

Subject:
- Extract short meaningful subject line
- DO NOT copy full paragraph

Important Dates:
- Extract ALL dates found
- Keep original format

Action Required:
- Extract ONLY actionable sentences:
  Examples:
    - "Please respond within 30 days"
    - "Cease communication"

---------------- DO NOT ----------------
- DO NOT guess
- DO NOT hallucinate sender/recipient
- DO NOT return paragraphs in Subject

---------------- DOCUMENT ----------------
{text[:1200]}
""")

In [225]:
@tool
def extract_business(text: str) -> str:
    """Extract Business document structured data."""
    return safe_llm(f"""
You are a HIGH PRECISION business document extractor.

Return ONLY valid JSON:

{{
  "Document Type": "",
  "Parties Involved": [],
  "Key Terms": [],
  "Dates": [],
  "Financial Amounts": [],
  "Obligations": []
}}

---------------- STRICT RULES ----------------

Document Type:
- Example: Agreement, Invoice, Contract
- Extract from title/header

Parties Involved:
- Extract ALL companies or individuals
- DO NOT include roles like "Client"
- CLEAN names only

Key Terms:
- Extract important clauses only
- MAX 5 entries
- SHORT phrases (not paragraphs)

Dates:
- Extract ALL dates

Financial Amounts:
- Extract ALL amounts ($, USD, etc.)

Obligations:
- Extract ONLY responsibilities:
  Examples:
    - "Party A shall pay..."
    - "Client must provide..."

---------------- DO NOT ----------------
- DO NOT guess
- DO NOT include long paragraphs
- DO NOT include duplicate entries

---------------- DOCUMENT ----------------
{text[:1200]}
""")

In [226]:
def validate_with_keys(text, doc_type):

    if doc_type == "LOA":
        keys = """
        - Client
        - Authorized Party
        - Authorization Scope
        - Effective Date
        - Signature
        """

        extra_rules = """
Authorizing Party (Client):
- MUST be a PERSON
- Can contain multiple names

Authorized Party:
- MUST be an ORGANIZATION
- Contains LLC / LAW / GROUP / FIRM

Signature:
- MUST exist
"""

    elif doc_type == "NOTICE":
        keys = """
        - Sender
        - Recipient
        - Subject
        - Action Required
        - Dates
        """

        extra_rules = """
DO NOT validate:
- Authorizing Party
- Signature

Sender:
- Must be person or organization

Dates:
- Can be in ANY language
"""

    elif doc_type == "BUSINESS":
        keys = """
        - Parties involved
        - Agreement type
        - Key terms
        - Dates
        - Amounts (if any)
        """

        extra_rules = """
DO NOT validate:
- Authorizing Party
- Signature

Parties:
- Can be organizations or individuals

Dates:
- Can be in ANY language
"""

    else:
        keys = "General structure only"
        extra_rules = ""

    return safe_llm(f"""
You are a document VALIDATION specialist.

The document may be in ANY language.

-------------------------------
VALIDATION CHECKS:
-------------------------------
1. Required fields present
2. Data format reasonable (not strict)
3. Dates logical (ANY language allowed)
4. Cross-field consistency

-------------------------------
IMPORTANT RULES:
-------------------------------
- Accept OCR noise if understandable
- Do NOT penalize minor issues
- NEVER infer or hallucinate values

{extra_rules}

-------------------------------
REQUIRED FIELDS:
-------------------------------
{keys}

-------------------------------
OUTPUT FORMAT:
-------------------------------
Issues:
- bullet points

Confidence: <number between 0 and 1>

DATA:
{text}

-------------------------------
STRICT RULES:
-------------------------------
- NO JSON
- NO markdown
- NO code blocks
- ONLY plain text
""")

In [227]:
def get_confidence(text):
    import re

    # Remove markdown stars
    clean = text.replace("*", "")

    match = re.search(
        r"confidence\s*[:\-]?\s*(0?\.\d+|1(?:\.0)?)",
        clean,
        re.IGNORECASE
    )

    if match:
        return float(match.group(1))

    print("Confidence not found → default 0.5")
    return 0.5

In [228]:
import json
import re

# ---------------- SAFE JSON PARSER ---------------- #
def safe_parse_json(text):
    try:
        return json.loads(text)
    except:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except:
                pass
    return {}


# ---------------- CLEAN USER INPUT PARSER ---------------- #
def parse_user_input(line):
    updates = {}

    if ":" in line:
        k, v = line.split(":", 1)

        key = k.strip().replace('"', '')
        value = v.strip().replace('"', '')

        updates[key] = value

    return updates


# ---------------- REVIEW ---------------- #
def human_review(doc, extracted, doc_type):

    # Using DICT
    current_dict = safe_parse_json(extracted)

    if not current_dict:
        print("Invalid extraction format. Skipping human review.")
        return extracted

    while True:

        print("\n Review Started")
        print("Document:", doc["file"])

        print("\n Current Extraction:\n")
        print(json.dumps(current_dict, indent=2))


        # ---------------- VALIDATION ---------------- #
        validation = validate_with_keys(
            json.dumps(current_dict),
            doc_type
        )

        confidence = round(get_confidence(validation), 2)


        # ---------------- CLEAN VALIDATION OUTPUT ---------------- #
        issues = ""

        if "Issues:" in validation:
            try:
                issues = validation.split("Issues:")[-1].split("Confidence")[0].strip()
            except:
                issues = ""

        print("\n Issues:")
        print(issues if issues else "None")

        print("\n Confidence:", confidence)

        # ---------------- AUTO ACCEPT ---------------- #
        if confidence >= 0.85:
            print("\n Auto-accepted after correction")
            #return json.dumps(current_dict, indent=2)
            return json.dumps(current_dict, indent=2), confidence
        # ---------------- USER ACTION ---------------- #
        print("\n Options:")
        print("1 → Accept")
        print("2 → Correction")

        choice = input("Enter: ").strip()

        # ---------------- ACCEPT ---------------- #
        if choice == "1":
            #return json.dumps(current_dict, indent=2)
            return json.dumps(current_dict, indent=2), confidence
        # ---------------- CORRECTION ---------------- #
        elif choice == "2":

            print("\n Enter corrections (key: value)")

            updates = {}

            while True:
                line = input().strip()

                if not line:
                    break

                parsed = parse_user_input(line)
                updates.update(parsed)

            if not updates:
                print("No valid updates provided")
                continue

            # ---------------- APPLY CORRECTIONS ---------------- #
            for k, v in updates.items():
                current_dict[k] = v

            # ---------------- MEMORY LEARNING ---------------- #
            for k, v in updates.items():
                if v and v.lower() != "not specified":
                    field_memory[doc_type].setdefault(k, []).append(v)

            print("\n Updated Extraction:")
            print(json.dumps(current_dict, indent=2))

            print("\n Updates Applied:")
            for k, v in updates.items():
                print(f"{k} → {v}")

            print("\n Re-validating...\n")

        else:
            print("Invalid choice")

In [229]:
def extract_client_from_bottom(text):
    lines = text.split("\n")[-40:]

    for line in lines:
        clean = re.sub(r'client name|co client name', '', line, flags=re.I)
        clean = clean.strip()

        # MUST be proper name (not label garbage)
        if re.search(r"[A-Z]{3,}", clean) and len(clean.split()) <= 4:
            clean = re.sub(r'[^A-Z ,]', '', clean)
            return clean.strip()

    return None

In [230]:
def fix_signature(text, extracted_json):
    data = safe_parse_json(extracted_json)

    if not data:
        return extracted_json

    # -------- CLEAN FUNCTION -------- #
    def clean_name(name):
        # HANDLE LIST (THIS FIXES YOUR ERROR)
        if isinstance(name, list):
            name = ", ".join(name)

        name = str(name)  # extra safety

        name = re.sub(r'[^A-Za-z ,]', '', name)
        name = re.sub(r'\s+', ' ', name)
        return name.strip()

    # -------- APPLY CLEANING -------- #
    client = clean_name(data.get("Authorizing Party", ""))
    sig = clean_name(data.get("Signature", ""))

    # -------- NORMALIZE -------- #
    def normalize(name):
        parts = [p.strip() for p in name.split(",") if p.strip()]
        return ", ".join(parts)

    client = normalize(client)
    sig = normalize(sig)

    # -------- REMOVE BAD SIGNATURE -------- #
    invalid = ["", "signature", "signature signature", "not specified"]

    if sig.lower() in invalid:
        sig = ""

    # -------- TRICT RULE -------- #
    # NEVER copy signature into client
    # NEVER modify client using signature

    # -------- REMOVE DUPLICATE -------- #
    if sig and client and sig.replace(" ", "").lower() == client.replace(" ", "").lower():
        sig = ""

    # -------- SAVE BACK -------- #
    data["Authorizing Party"] = client
    data["Signature"] = sig

    return json.dumps(data)

In [231]:
def process_document(doc):

    print("\n" + "="*80)
    print(f"📄 Processing: {doc['file']}")

    text = doc["text"]

    # RULE BASED FIRST
    doc_type, cls_conf = rule_based_classification(text)

    # FALLBACK TO LLM
    if doc_type == "UNKNOWN":
        cls_output = classify_document.invoke(text)
        doc_type, cls_conf = parse_classification(cls_output)

    print(f"\nType: {doc_type} ({cls_conf})")

    # FORCE LOA if keyword
    if re.search(r"power of attorney|authorize|retained the services", text, re.I):
        print("Override → LOA")
        doc_type = "LOA"

    # MEMORY
    memory_hint = json.dumps(field_memory.get(doc_type, {}))[:500]

    client_hint = extract_client_from_bottom(text)
    if client_hint:
      text += f"\n\nIMPORTANT: CLIENT NAME IS: {client_hint}\nUSE THIS VALUE FOR Authorizing Party"

    # EXTRACTION
    if doc_type == "LOA":
        extracted = extract_loa.invoke(text + f"\n\nLearnings:\n{memory_hint}")
    elif doc_type == "NOTICE":
        extracted = extract_notice.invoke(text + f"\n\nLearnings:\n{memory_hint}")
    else:
        extracted = extract_business.invoke(text + f"\n\nLearnings:\n{memory_hint}")

    print("\n Extraction:\n", extracted)

    # APPLY SIGNATURE FIX
    # APPLY SIGNATURE FIX
    extracted = fix_signature(text, extracted)

    # REMOVE LOA FIELDS FOR NON-LOA
    data = safe_parse_json(extracted)

    if doc_type in ["NOTICE", "BUSINESS"]:
        data.pop("Authorizing Party", None)
        data.pop("Signature", None)

    extracted = json.dumps(data)

    # VALIDATE
    validation = validate_with_keys(extracted, doc_type)
    confidence = round(get_confidence(validation), 2)

    print("\n Confidence:", confidence)

    # HUMAN LOOP
    if confidence < 0.85:
        final, confidence = human_review(doc, extracted, doc_type)
    else:
        print("\n Auto Approved")
        final = extracted

    save_result(doc, doc_type, confidence, final)

    print("\n Saved")

In [232]:
conn = sqlite3.connect("documents.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS documents (
id INTEGER PRIMARY KEY,
file TEXT,
type TEXT,
confidence REAL,
output TEXT
)
""")
conn.commit()

In [233]:
def save_result(doc, doc_type, confidence, output):

    if isinstance(output, (dict, list, tuple)):
        output = json.dumps(output)

    elif isinstance(output, str):
        try:
            parsed = json.loads(output)
            output = json.dumps(parsed)
        except:
            output = json.dumps({"raw_output": output})

    else:
        output = json.dumps({"raw_output": str(output)})

    cur.execute(
        "INSERT INTO documents (file,type,confidence,output) VALUES (?,?,?,?)",
        (doc["file"], doc_type, confidence, output)
    )
    conn.commit()

In [234]:
#folder_path = "/content/drive/MyDrive/pdf"

docs = load_all_pdfs(folder_path)

for doc in docs:
    process_document(doc)
    time.sleep(3)


📄 Loading: LOA2.pdf

📄 Loading: LOA3.pdf

📄 Loading: LOA4.pdf

📄 Loading: LOA5.pdf

📄 Loading: LOA6.pdf

📄 Loading: LOA7.pdf

📄 Loading: LOA8.pdf

📄 Loading: LOA9.pdf

📄 Loading: LoA1.pdf

📄 Loading: bw_doc_1.pdf
Low text → using OCR
OCR running: /content/drive/MyDrive/pdf/bw_doc_1.pdf

📄 Loading: bw_doc_2.pdf
Low text → using OCR
OCR running: /content/drive/MyDrive/pdf/bw_doc_2.pdf

📄 Loading: bw_doc_3.pdf
Low text → using OCR
OCR running: /content/drive/MyDrive/pdf/bw_doc_3.pdf

📄 Loading: bw_doc_4.pdf
Low text → using OCR
OCR running: /content/drive/MyDrive/pdf/bw_doc_4.pdf

📄 Loading: bw_doc_5.pdf
Low text → using OCR
OCR running: /content/drive/MyDrive/pdf/bw_doc_5.pdf

📄 Loading: notice_1.pdf
Low text → using OCR
OCR running: /content/drive/MyDrive/pdf/notice_1.pdf

📄 Loading: notice_2.pdf
Low text → using OCR
OCR running: /content/drive/MyDrive/pdf/notice_2.pdf

📄 Loading: notice_3.pdf
Low text → using OCR
OCR running: /content/drive/MyDrive/pdf/notice_3.pdf

📄 Loading: notice_

In [236]:
import json

cur.execute("SELECT * FROM documents")

rows = cur.fetchall()

for row in rows:
    print("\n============================")
    print(f"File: {row[1]}")
    print(f"Type: {row[2]}")
    print(f"Confidence: {row[3]}")

    try:
        parsed = json.loads(row[4])
        print(json.dumps(parsed, indent=2))
    except:
        print(row[4])


File: LOA2.pdf
Type: LOA
Confidence: 0.8
{
  "Authorizing Party": "LOVETTA CANNUNZIATA",
  "Authorized Party": "LAW OFFICES OF DONALD A. GREEN, APLC",
  "Authorization Scope": [
    "release to my attorney and agent, all financial records, confidential and otherwise and other data pertaining to my above\u00adreferenced account;",
    "review my account history with my attorney;",
    "discuss my account in all respects with my attorney;",
    "negotiate all matters pertaining to my account;",
    "make and receive offers of settlement;",
    "reach an accord and satisfaction of my debts;",
    "do every act, deed, and thing necessary to be done, in order to carry out the above authorized acts, as if I was personally present and so acting."
  ],
  "Effective Date": "7/2/2025",
  "Signature": ""
}

File: LOA3.pdf
Type: LOA
Confidence: 0.8
{
  "Authorizing Party": "BELLINA DAVANZO AMOEDO, RADCLIFF",
  "Authorized Party": "Five Lakes Law Group PLLC",
  "Authorization Scope": [
    "releas